In [1]:
!pip install pandas pyodbc nltk 



Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [2]:
%pip install pyodbc

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd 
import pyodbc
import nltk 
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [4]:
nltk.download('vader_lexicon')

def fetch_data_from_sql():
    conn_str = (
        "Driver={ODBC Driver 17 for SQL Server};"
        "Server=localhost;"
        "Database=PortfolioProject_MarketingAnalytics;"
        "Trusted_Connection=yes;"
        "TrustServerCertificate=yes;"

    )

    conn = pyodbc.connect(conn_str)
    query = "Select ReviewID, CustomerID, ProductID, ReviewDate, ReviewText, Rating from dbo.customer_reviews"

    df = pd.read_sql(query, conn)
    conn.close()
    return df 
   

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Pratham\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [5]:
customer_reviews_df = fetch_data_from_sql()

C:\Users\Pratham\AppData\Local\Temp\ipykernel_28236\2856985770.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [6]:
customer_reviews_df  

,ReviewID,CustomerID,ProductID,ReviewDate,ReviewText,Rating
0,1,77,18,2023-12-23,"Average experience, nothing special.",3
1,2,80,19,2024-12-25,The quality is top-notch.,5
2,3,50,13,2025-01-26,Five stars for the quick delivery.,4
3,4,78,15,2025-04-21,"Good quality, but could be cheaper.",3
4,5,64,2,2023-07-16,"Average experience, nothing special.",3
...,...,...,...,...,...,...
1358,1359,28,4,2023-05-25,Not worth the money.,3
1359,1360,58,12,2023-11-13,"Average experience, nothing special.",2
1360,1361,96,15,2023-03-07,Customer support was very helpful.,5
1361,1362,99,2,2025-12-03,Product did not meet my expectations.,1


In [7]:
sia = SentimentIntensityAnalyzer()

In [8]:
def analyze_sentiment(review):
    sentiment = sia.polarity_scores(review)
    return sentiment['compound'] # returns the compound score which is a normalized score between -1 and +1 


In [9]:
def categorize_sentiment(score, rating):
    # Use both the text sentiment score and the numerical rating to determine sentiment category
    if score > 0.05:  # Positive sentiment score
        if rating >= 4:
            return 'Positive'  # High rating and positive sentiment
        elif rating == 3:
            return 'Mixed Positive'  # Neutral rating but positive sentiment
        else:
            return 'Mixed Negative'  # Low rating but positive sentiment
    elif score < -0.05:  # Negative sentiment score
        if rating <= 2:
            return 'Negative'  # Low rating and negative sentiment
        elif rating == 3:
            return 'Mixed Negative'  # Neutral rating but negative sentiment
        else:
            return 'Mixed Positive'  # High rating but negative sentiment
    else:  # Neutral sentiment score
        if rating >= 4:
            return 'Positive'  # High rating with neutral sentiment
        elif rating <= 2:
            return 'Negative'  # Low rating with neutral sentiment
        else:
            return 'Neutral'  # Neutral rating and neutral sentiment
                

In [10]:
def sentiment_bucket(score):
    if score >= 0.5:
        return '0.5 to 1.0'  # Strongly positive sentiment
    elif 0.0 <= score < 0.5:
        return '0.0 to 0.49'  # Mildly positive sentiment
    elif -0.5 <= score < 0.0:
        return '-0.49 to 0.0'  # Mildly negative sentiment
    else:
        return '-1.0 to -0.5'  # Strongly negative sentiment

In [11]:
customer_reviews_df['SentimentScore'] = customer_reviews_df['ReviewText'].apply(analyze_sentiment)

In [12]:
customer_reviews_df 

,ReviewID,CustomerID,ProductID,ReviewDate,ReviewText,Rating,SentimentScore
0,1,77,18,2023-12-23,"Average experience, nothing special.",3,-0.3089
1,2,80,19,2024-12-25,The quality is top-notch.,5,0.0000
2,3,50,13,2025-01-26,Five stars for the quick delivery.,4,0.0000
3,4,78,15,2025-04-21,"Good quality, but could be cheaper.",3,0.2382
4,5,64,2,2023-07-16,"Average experience, nothing special.",3,-0.3089
...,...,...,...,...,...,...,...
1358,1359,28,4,2023-05-25,Not worth the money.,3,-0.1695
1359,1360,58,12,2023-11-13,"Average experience, nothing special.",2,-0.3089
1360,1361,96,15,2023-03-07,Customer support was very helpful.,5,0.6997
1361,1362,99,2,2025-12-03,Product did not meet my expectations.,1,0.0000


In [13]:
customer_reviews_df['SentimentCategory'] = customer_reviews_df.apply(
    lambda row: categorize_sentiment(
        row['SentimentScore'], row['Rating']
    ),
    axis = 1
)

In [14]:
customer_reviews_df['SentimentBucket'] = customer_reviews_df['SentimentScore'].apply(sentiment_bucket)

In [15]:
customer_reviews_df 

,ReviewID,CustomerID,ProductID,ReviewDate,ReviewText,Rating,SentimentScore,SentimentCategory,SentimentBucket
0,1,77,18,2023-12-23,"Average experience, nothing special.",3,-0.3089,Mixed Negative,-0.49 to 0.0
1,2,80,19,2024-12-25,The quality is top-notch.,5,0.0000,Positive,0.0 to 0.49
2,3,50,13,2025-01-26,Five stars for the quick delivery.,4,0.0000,Positive,0.0 to 0.49
3,4,78,15,2025-04-21,"Good quality, but could be cheaper.",3,0.2382,Mixed Positive,0.0 to 0.49
4,5,64,2,2023-07-16,"Average experience, nothing special.",3,-0.3089,Mixed Negative,-0.49 to 0.0
...,...,...,...,...,...,...,...,...,...
1358,1359,28,4,2023-05-25,Not worth the money.,3,-0.1695,Mixed Negative,-0.49 to 0.0
1359,1360,58,12,2023-11-13,"Average experience, nothing special.",2,-0.3089,Negative,-0.49 to 0.0
1360,1361,96,15,2023-03-07,Customer support was very helpful.,5,0.6997,Positive,0.5 to 1.0
1361,1362,99,2,2025-12-03,Product did not meet my expectations.,1,0.0000,Negative,0.0 to 0.49


In [17]:
customer_reviews_df.to_csv('fact_customer_reviews_with_sentiment.csv',index = False)